# DeepLabV3+ Training & Inference Notebook

Notebook ini untuk melatih model DeepLabV3+ pada dataset Plant Phenotyping (20 kelas) menggunakan arsitektur dari `models/`.

**Environment:** VS Code + Colab Kernel (GPU Colab)
**Dataset:** Downloaded via `data/download_dataset.py` (kagglehub)
**Model:** DeepLabV3+ dari `models/deeplab.py` dengan backbone ResNet/Xception/DRN/MobileNet
**Output:** Checkpoint `.pth.tar` di folder `experiments/`

## 0. Setup Environment (Colab-specific)

In [1]:
import os, shutil, sys
from pathlib import Path

REPO_URL = 'https://github.com/adinmusababa/segmentasi.git'
REPO_DIR = Path('/content/segmentasi')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
!git clone -b setup {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
print(f'CWD: {os.getcwd()}')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

REPO_PATH = REPO_DIR  # alias global untuk sel-sel berikutnya
IN_COLAB = True

Cloning into '/content/segmentasi'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 126 (delta 37), reused 124 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (126/126), 1.55 MiB | 4.60 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/segmentasi
CWD: /content/segmentasi


In [2]:
# Download dataset
!python data/download_dataset.py

python3: can't open file '/content/segmentasi/data/download_dataset.py': [Errno 2] No such file or directory


In [3]:
# # Colab environment setup
# import sys
# import os
# from pathlib import Path

# # Detect if running in Colab
# IN_COLAB = 'google.colab' in sys.modules
# print(f"IN_COLAB: {IN_COLAB}")

# # NOTE: Semua data (repo + dataset) di-simpan di session runtime Colab (/content),
# # yang bersifat sementara dan hilang saat runtime di-reset/disconnect.
# # Ini sesuai preferensi: TIDAK menyimpan ke Google Drive.
# # Kalau mau file hasil (dataset, checkpoint) tahan lama, simpan manual ke Drive.

# if IN_COLAB:
#     # Clone repo ke session runtime (bukan Drive)
#     REPO_PATH = Path('/content/deeplabV3-PyTorch')
#     if not REPO_PATH.exists():
#         print("Cloning repository...")
#         !git clone https://github.com/adinmusababa/deeplabV3-PyTorch.git /content/deeplabV3-PyTorch
#     os.chdir(REPO_PATH)
#     print(f"Working dir: {os.getcwd()}")
# else:
#     # Local/VS Code: assume already in repo root
#     REPO_PATH = Path.cwd()
#     while not (REPO_PATH / 'models').exists() and REPO_PATH != REPO_PATH.parent:
#         REPO_PATH = REPO_PATH.parent
#     os.chdir(REPO_PATH)
#     print(f"Working dir: {os.getcwd()}")

# # Add project root to sys.path
# if str(REPO_PATH) not in sys.path:
#     sys.path.insert(0, str(REPO_PATH))

# # Verify structure
# print("models/ exists:", (REPO_PATH / "models").exists())
# print("data/ exists:", (REPO_PATH / "data").exists())
# print("configs/ exists:", (REPO_PATH / "configs").exists())
# print("kagglehub cache di: /root/.cache/kagglehub (session temp, bukan Drive)")

## 1. Install Dependencies

In [7]:
# Install requirements
!pip install -q kagglehub pyyaml tensorboardX tqdm scikit-learn matplotlib pillow numpy torch torchvision

# Verify torch CUDA
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


## 2. Download & Organize Dataset

In [ ]:
# # Run download script
# import subprocess
# result = subprocess.run([sys.executable, "data/download_dataset.py"], capture_output=True, text=True)
# print(result.stdout)
# if result.stderr:
#     print("STDERR:", result.stderr)

# # Verify
# from pathlib import Path
# imgs = list((REPO_PATH / "data" / "imgs").glob("*.png"))
# masks = list((REPO_PATH / "data" / "masks").glob("*.png"))
# print(f"Images: {len(imgs)}")
# print(f"Masks: {len(masks)}")
# if imgs:
#     print(f"First few: {[f.name for f in imgs[:5]]}")

## 3. Configure Training (EDIT HERE)

In [9]:
import torch
import yaml
from pathlib import Path

# Load base config
with open("configs/config.yml") as f:
    config = yaml.safe_load(f)

# ========== OVERRIDE FOR PLANT DATASET ==========
config["dataset"]["base_path"] = str(REPO_PATH)  # root project
config["dataset"]["dataset_name"] = "plant_phenotyping"
# Plant dataset: background(0) + 19 plant organ classes = 20
config["network"]["num_classes"] = 20
config["network"]["backbone"] = "resnet"  # pilihan: resnet, xception, drn, mobilenet
config["network"]["sync_bn"] = False  # True hanya kalau multi-GPU
config["network"]["freeze_bn"] = False
config["network"]["use_cuda"] = torch.cuda.is_available()

config["image"]["out_stride"] = 16
config["image"]["base_size"] = 513
config["image"]["crop_size"] = 513  # turunkan ke 256/320 untuk eksperimen cepat

config["training"]["workers"] = 4 if torch.cuda.is_available() else 0
config["training"]["batch_size"] = 4 if torch.cuda.is_available() else 2  # minimal 2 untuk BatchNorm
config["training"]["epochs"] = 30  # ubah sesuai kebutuhan
config["training"]["start_epoch"] = 0
config["training"]["lr"] = 0.0005
config["training"]["lr_scheduler"] = "poly"  # poly, step, cos
config["training"]["momentum"] = 0.9
config["training"]["weight_decay"] = 0.0005
config["training"]["nesterov"] = False
config["training"]["loss_type"] = "ce"  # ce atau focal
config["training"]["use_balanced_weights"] = False
config["training"]["no_val"] = False
config["training"]["val_interval"] = 1
config["training"]["train_on_subset"]["enabled"] = False  # True untuk quick test
config["training"]["train_on_subset"]["dataset_fraction"] = 0.1

# Resume training (optional)
config["training"]["weights_initialization"]["use_pretrained_weights"] = False  # True kalau mau resume
config["training"]["weights_initialization"]["restore_from"] = "./experiments/checkpoint_last.pth.tar"

config["training"]["model_best_checkpoint"]["enabled"] = True
config["training"]["model_best_checkpoint"]["out_file"] = "./experiments/checkpoint_best.pth.tar"
config["training"]["model_last_checkpoint"]["enabled"] = True
config["training"]["model_last_checkpoint"]["out_file"] = "./experiments/checkpoint_last.pth.tar"
# Saver uses ./experiments/ directory (hardcoded in utils/saver.py)

config["training"]["tensorboard"]["enabled"] = True
config["training"]["tensorboard"]["log_dir"] = "./tensorboard/"

# Seed for reproducibility
config["seed"] = 42

# Save modified config
config_path = REPO_PATH / "configs" / "config_plant.yml"
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Config saved to: {config_path}")
print("Key settings:")
print(f"  num_classes: {config["network"]["num_classes"]}")
print(f"  backbone: {config["network"]["backbone"]}")
print(f"  batch_size: {config["training"]["batch_size"]}")
print(f"  epochs: {config["training"]["epochs"]}")
print(f"  crop_size: {config["image"]["crop_size"]}")
print(f"  use_cuda: {config["network"]["use_cuda"]}")

Config saved to: /content/segmentasi/configs/config_plant.yml
Key settings:
  num_classes: 20
  backbone: resnet
  batch_size: 4
  epochs: 30
  crop_size: 513
  use_cuda: True


## 4. Training

In [14]:
# Import Trainer
from trainers.trainer import Trainer

# Buat direktori experiments jika belum ada (untuk checkpoint)
(REPO_PATH / "experiments").mkdir(parents=True, exist_ok=True)

# checkname diperlukan oleh Trainer/Saver
config["checkname"] = "deeplab-" + str(config["network"]["backbone"])

# Initialize trainer
trainer = Trainer(config)

print(f"Starting Epoch: {trainer.config["training"]["start_epoch"]}")
print(f"Total Epochs: {trainer.config["training"]["epochs"]}")
print(f"Train loader: {len(trainer.train_loader)} batches")
print(f"Val loader: {len(trainer.val_loader)} batches")
print(f"Test loader: {len(trainer.test_loader)} batches")
print(f"Classes: {trainer.nclass}")

Using poly LR Scheduler!
Starting Epoch: 0
Total Epochs: 30
Train loader: 70 batches
Val loader: 34 batches
Test loader: 34 batches
Classes: 20


In [15]:
# Run training loop
for epoch in range(trainer.config['training']['start_epoch'], trainer.config['training']['epochs']):
    trainer.training(epoch)
    if not trainer.config['training']['no_val'] and epoch % config['training']['val_interval'] == (config['training']['val_interval'] - 1):
        trainer.validation(epoch)

trainer.writer.close()
print("Training completed!")

  0%|          | 0/70 [00:00<?, ?it/s]


=>Epoches 0, learning rate = 0.0005,                 previous best = 0.0000


Train loss: 0.791:   0%|          | 0/70 [00:02<?, ?it/s]


TypeError: make_grid() got an unexpected keyword argument 'range'

## 5. Load Best Model for Inference

In [ ]:
# Load predictor with best checkpoint
from predictors.predictor import Predictor

checkpoint_path = './experiments/checkpoint_best.pth.tar'
if not Path(checkpoint_path).exists():
    checkpoint_path = './experiments/checkpoint_last.pth.tar'
    print(f"Best not found, using last: {checkpoint_path}")
else:
    print(f"Using best checkpoint: {checkpoint_path}")

predictor = Predictor(config, checkpoint_path=checkpoint_path)
print(f"Model loaded. Classes: {predictor.num_classes}")

## 6. Inference on Single Image

In [ ]:
# Test on a sample image from dataset
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Pick first image from data/imgs
test_images = list((REPO_PATH / "data" / "imgs").glob("*.png"))
if test_images:
    test_img = str(test_images[0])
    print(f"Testing on: {test_img}")
    
    image, prediction = predictor.segment_image(test_img)
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image.astype(np.uint8))
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    
    # Prediction mask
    im1 = axes[1].imshow(prediction, cmap='nipy_spectral', vmin=0, vmax=predictor.num_classes-1)
    axes[1].set_title("Prediction Mask")
    axes[1].axis('off')
    
    # Overlay
    overlay = image.copy()
    # Create colormap
    colors = np.random.RandomState(42).randint(0, 255, (predictor.num_classes, 3)).astype(np.uint8)
    colors[0] = [0, 0, 0]  # background black
    pred_colored = colors[prediction]
    overlay = (overlay * 0.6 + pred_colored * 0.4).astype(np.uint8)
    axes[2].imshow(overlay)
    axes[2].set_title("Overlay (60% img + 40% mask)")
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Prediction shape: {prediction.shape}")
    print(f"Unique classes predicted: {np.unique(prediction)}")
else:
    print("No test images found in data/imgs/")

## 7. Batch Inference on Test Set (Evaluation)

In [5]:
# Run evaluation on test set
predictor.inference_on_test_set()

## 8. Batch Inference on Folder (Save Predictions)

In [ ]:
# Save predictions for all images in a folder
from pathlib import Path
from tqdm import tqdm

INPUT_DIR = REPO_PATH / "data" / "imgs"
OUTPUT_DIR = REPO_PATH / "inference_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

img_files = sorted(list(INPUT_DIR.glob("*.png")))
print(f"Processing {len(img_files)} images...")

for img_path in tqdm(img_files):
    try:
        _, prediction = predictor.segment_image(str(img_path))
        out_path = OUTPUT_DIR / f"{img_path.stem}_pred.png"
        Image.fromarray(prediction.astype(np.uint8)).save(out_path)
    except Exception as e:
        print(f"Error on {img_path.name}: {e}")

print(f"Done! Results saved to: {OUTPUT_DIR}")

## 9. TensorBoard (Optional)

In [ ]:
# Launch TensorBoard in Colab
if IN_COLAB:
    %load_ext tensorboard
    %tensorboard --logdir ./tensorboard --port 6006
else:
    print("Run locally: tensorboard --logdir ./tensorboard")

## 10. Tips & Next Steps

- **Cepatkan eksperimen:** turunkan `crop_size` ke 256/320, `epochs` ke 5-10, `train_on_subset.enabled: true`
- **Ganti backbone:** `mobilenet` atau `xception` lebih cepat dari `resnet`
- **Resume training:** set `weights_initialization.use_pretrained_weights: true` dan `start_epoch`
- **Class weights:** enable `use_balanced_weights: true` untuk dataset tidak seimbang
- **Multi-GPU:** set `sync_bn: true` dan `use_cuda: true` (Colab Pro+ dengan multi-GPU)
- **Checkpoint format:** `.pth.tar` standar PyTorch, bisa di-load di `main.py` atau script custom